In [13]:
import pandas as pd
import numpy as np
import math
import nltk
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from collections import Counter
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from sklearn.preprocessing import MinMaxScaler

In [2]:
tqdm.pandas(desc="Calcul N-Gram Rarity")

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/augustin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/augustin/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
df_wiki_0 = pd.read_parquet("../wiki/train-00000-of-00004.parquet")
df_wiki_1 = pd.read_parquet("../wiki/train-00001-of-00004.parquet")
df_wiki_2 = pd.read_parquet("../wiki/train-00002-of-00004.parquet")
df_wiki = pd.concat([df_wiki_0, df_wiki_1, df_wiki_2])

df_reactions = pd.read_parquet("../reactions/reactions.parquet")

In [4]:
reference_bigrams = Counter()
total_reference_bigrams = 0

for text in tqdm(df_wiki['text'], desc="Analyse Wikipédia"):
    tokens = [word.lower() for word in word_tokenize(text, language='french') if word.isalpha()]
    
    bigrams = list(ngrams(tokens, 2))
    
    reference_bigrams.update(bigrams)
    total_reference_bigrams += len(bigrams)

min_prob = 1 / (total_reference_bigrams + 100000)
penalty_score = -math.log(min_prob)

print(f"Dictionnaire prêt : {len(reference_bigrams)} bigrammes uniques trouvés.")

Analyse Wikipédia:   0%|          | 0/187500 [00:00<?, ?it/s]

Dictionnaire prêt : 2863661 bigrammes uniques trouvés.


In [5]:
count_pen = 0
count_ngram = 0

In [6]:
def calculate_ngram_rarity(text, n=2):
    """
    Calcule le score de rareté moyen des n-grammes d'un texte.
    """
    global count_pen
    global count_ngram
    if not isinstance(text, str) or len(text.strip()) == 0:
        return np.nan
        
    tokens = [word.lower() for word in word_tokenize(text, language='french') if word.isalpha()]
    
    if len(tokens) < n:
        return np.nan
        
    text_ngrams = list(ngrams(tokens, n))
    rarity_scores = []
    
    for gram in text_ngrams:
        count = reference_bigrams.get(gram, 0)
        if count > 0:
            prob = count / total_reference_bigrams
            rarity_scores.append(-math.log(prob))
        else:
            # Si le bigramme est totalement inconnu du corpus de référence,
            # on lui donne le score de pénalité maximal (très rare = très créatif/surprenant)

            rarity_scores.append(penalty_score)
            count_pen += 1
    
    count_ngram += len(text_ngrams)
            
    return np.mean(rarity_scores)

In [7]:
df_reactions['ngram_rarity_raw'] = df_reactions["response_content"].progress_apply(calculate_ngram_rarity)

Calcul N-Gram Rarity:   0%|          | 0/94294 [00:00<?, ?it/s]

In [8]:
count_pen/count_ngram*100

33.63322815927407

In [ ]:
df_eval = df_reactions.dropna(subset=['ngram_rarity_raw', 'creative']).copy()
df_eval = df_eval[df_eval['creative'].isin([True, False])].copy()

df_eval['creative'] = df_eval['creative'].astype(int)


corr, p_val = pearsonr(df_eval['ngram_rarity_raw'], df_eval['creative'])

print(f"Corrélation de Pearson : {corr:.3f} (p-value: {p_val:.4E})")

Corrélation de Pearson : 0.054 (p-value: 1.5462E-60)


In [15]:
scaler = MinMaxScaler()
df_metrics = df_eval[["id"]]
df_metrics["ngram_rarity"] = scaler.fit_transform(df_eval[["ngram_rarity_raw"]])


In [17]:
save_path = "reactions_with_ngram.parquet"
df_metrics.to_parquet(save_path, index=False)

print(f"Sauvegarde réussie ! {len(df_metrics)} lignes stockées dans '{save_path}'.")

Sauvegarde réussie ! 93233 lignes stockées dans 'reactions_with_ngram.parquet'.
